### The aim of this notebook is to perform Time Series analysis on a set of transaction data & also perform association-rule based analysis.

### Do read and upvote.! 

In [ ]:
import numpy as np 
import pandas as pd
import os
print(os.listdir("../input"))

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
color = sns.color_palette()

%matplotlib inline

from plotly import tools
import plotly.offline as py
py.init_notebook_mode(connected=True)
import plotly.graph_objs as go

In [ ]:
data = pd.read_csv("../input/BreadBasket_DMS.csv")
data.shape

### A peek at the day-to-day transactions

In [ ]:
data.head(10)

### Data pre-processing

In [ ]:
data['Date'] = pd.to_datetime(data['Date'],format='%Y-%m-%d')

In [ ]:
data['Time'] = pd.to_datetime(data['Time'])
data['Times'] = data['Time'].dt.time

In [ ]:
#tot_tr = data.groupby('Date', as_index=True)['Transaction'].sum().reset_index()
tot_tr1 = data.groupby(['Date', 'Transaction']).size().reset_index()
tot_tr1.columns = ['Date', 'Transaction', 'count']

In [ ]:
tot_tr = tot_tr1.groupby('Date', as_index=True)['count'].sum().reset_index()
tot_tr.columns = ['Date', 'Transaction']

In [ ]:
tot1 = tot_tr.iloc[:56]
tot2 = tot_tr.iloc[56:]

In [ ]:
b = pd.to_datetime('2017-01-02 00:00:00',format='%Y-%m-%d')
c = pd.to_datetime('2016-12-25 00:00:00',format='%Y-%m-%d')
d = pd.to_datetime('2016-12-26 00:00:00',format='%Y-%m-%d')

In [ ]:
tot2.loc[-2] = [c, 0]
tot2.loc[-1] = [d, 0] 
tot2.index = tot2.index + 2
tot2 = tot2.sort_index()
tot2 = tot2.rename(index={0: 56, 1:57})

In [ ]:
#tot1.append(tot2)
tot_tr2 = tot1.append(tot2)

In [ ]:
tot1 = tot_tr2.iloc[:64]
tot2 = tot_tr2.iloc[64:]

In [ ]:
tot2.loc[-1] = [b, 0]
tot2.index = tot2.index + 1
tot2 = tot2.sort_index()
tot2 = tot2.rename(index={0:64})

In [ ]:
#tot1.append(tot2)
tot_tr3 = tot1.append(tot2)

In [ ]:
tot_tr3 = tot_tr3.replace({'Transaction': {0: 1}})

### Trend of transactions - Which day had the most number of transactions?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
color = sns.color_palette()

%matplotlib inline

from plotly import tools
import plotly.offline as py
py.init_notebook_mode(connected=True)
import plotly.graph_objs as go

xtr = tot_tr.loc[tot_tr['Transaction'].idxmax()][0]
ytr = tot_tr['Transaction'].max()

data1 = [go.Scatter(
          x=tot_tr.Date,
          y=tot_tr.Transaction)]

layout = go.Layout(
   showlegend=False,
    annotations=[
        dict(
            x=xtr,
            y=ytr,
            xref='x',
            yref='y',
            text='Highest',
            showarrow=True,
            arrowhead=7,
            ax=0,
            ay=-40
        )
    ]
)

fig = go.Figure(data=data1, layout=layout)
py.iplot(fig, filename='multiple-annotation')

### We can see in the above graph that, Feb 4th had the most number of transactions

### Which hour had the most transactions?

In [ ]:
data['hour'] = data['Time'].dt.hour
#data.head(10)

In [ ]:
s = data['hour'].value_counts().reset_index()

In [ ]:
plt.rcParams['figure.figsize']=(20,20)
g = sns.jointplot(x=s['index'], y=s['hour'], data=s, kind="kde", color = "m", size=12, aspect=3);
g.plot_joint(plt.scatter, c="w", s=30, linewidth=1, marker="+")
g.ax_joint.collections[0].set_alpha(0)
g.set_axis_labels("Hour", "Count of transactions");

### As seen in the plot above, its between 11AM & 3 PM where most transactions happened in any given day.

### What about the items bought?

In [ ]:
item_cnt = data['Item'].value_counts().reset_index()
item_cnt.columns = ['Item', 'Count']
item_cnt = item_cnt[item_cnt.Item != 'NONE']
item_cnt = item_cnt.head(15)
#item_cnt

In [ ]:
import plotly.graph_objs as go

labels = item_cnt['Item'].values.tolist()
values = item_cnt['Count'].values.tolist()

trace = go.Pie(labels=labels, values=values)

py.iplot([trace], filename='item_chart')

### Among the top 15 items bought, coffee has been bought nearly 33% of the time

In [ ]:
item_cnt = data.groupby(['Item', 'hour']).size().reset_index()
item_cnt.columns = ['Item', 'Hour', 'Count']
item_cnt = item_cnt[item_cnt.Item != 'NONE']
item_cnt = item_cnt.sort_values(by='Count', ascending=False)
#item_cnt#.head(100)

### Among the top 3 items bought, lets see at what time of the day they are normally purchased

In [ ]:
item_cnt_cf = data.groupby(['Item', 'hour']).size().reset_index()
item_cnt_cf.columns = ['Item', 'Hour', 'Count']
item_cnt_cf = item_cnt[item_cnt['Item'].isin(['Coffee', 'Bread', 'Tea'])]
#item_cnt_cf

In [ ]:
g = sns.PairGrid(item_cnt_cf, hue="Item", height=10, aspect=1)
g.map_diag(plt.hist)
g.map_offdiag(plt.scatter)
g.add_legend();

### As seen in the plot above, Coffee & tea is mostly during the entire day, a little scattered, while bread is bought during the morning hours mostly.

In [ ]:
item_cnt_cake = data.groupby(['Item', 'hour']).size().reset_index()
item_cnt_cake.columns = ['Item', 'Hour', 'Count']
item_cnt_cake = item_cnt[item_cnt['Item'].isin(['Cake', 'Pastry', 'Sandwich', 'Medialuna'])]

g = sns.PairGrid(item_cnt_cake, hue="Item", height=10, aspect=1)
g.map_diag(plt.hist)
g.map_offdiag(plt.scatter)
g.add_legend();

### Now, to implement few techniques in this kernel, lets split the dataset we have into train and test

In [ ]:
train= tot_tr3[0:100] 
test= tot_tr3[100:]

In [ ]:
train_series = pd.Series(train.Transaction.values, index=pd.date_range(train.Date.min(),train.Date.max(),freq='D'))
test_series = pd.Series(test.Transaction.values, index=pd.date_range(test.Date.min(),test.Date.max(),freq='D'))

In [ ]:
from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 18, 12
plt.plot(train_series, label='train')
plt.plot(test_series, label='test')
plt.title('Train and test Graph')
plt.legend()
plt.show()

### Time Series Components

***
The time series at hand is data related to bakery transactions.

Let’s now try to deconstruct various components that make up the time series at hand. A time
series is said to be comprised of the following three major components:

* Seasonality: These are the periodic fluctuations in the observed data. 
* Trend: This is the increasing or decreasing behavior of the series with time.
*  Residual: This is the remaining signal after removing the seasonality and trend
signals. It can be further decomposed to remove the noise component as well.
***

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
ts_trs = pd.Series(tot_tr3.Transaction.values, index=pd.date_range(tot_tr3.Date.min(),tot_tr3.Date.max(),freq='D'))
deompose = seasonal_decompose(ts_trs, freq=24)
from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 15, 10
deompose.plot()

The time series has both upward and downward trend. It shows a decreasing trend and then an increasing trend as we saw in the initial plot as well.

The series certainly has a monthly periodicity or seasonality to it. The remaining signal is what is marked as residual.

### Smoothing techniques

Why Smoothing? 

Time series data have inherent dependency on historical observations and there are multiple factors impacting each observation.
It is an inherent property of time series data to have random variation to it apart from its other constituents. 

To better understand, model, and utilize time series for prediction related tasks, lets perform smoothing.

Smoothing helps reduce the effect of random variation and helps clearly reveal the seasonality, trend, and residual components of the series. There are various methods
to smooth out a time series. 

####  Moving Average


Instead of taking an average of complete time series to summarize, moving average makes use of a rolling windowed approach. In this case, we compute the mean
of each successive smaller windows of past data to smoothen out the impact of random variation. 

### Smoothening for the entire set

#### 3 month moving average

In [ ]:
tot_tr3['moving_average'] = tot_tr3['Transaction'].rolling(window=3, center=False).mean()
plt.figure(figsize=(20,10))
plt.plot(tot_tr3.Date, tot_tr3.Transaction,'-',color='black',alpha=0.3)
plt.plot(tot_tr3.Date, tot_tr3.moving_average,color='b')
plt.title('Transaction and Moving Average Smoothening')
plt.legend()
plt.show()

### Train - test split and prediction 

In [ ]:
y_hat_avg = test.copy()
y_hat_avg['moving_avg_forecast'] = train['Transaction'].rolling(60).mean().iloc[-1]
plt.figure(figsize=(18,12))
plt.plot(train['Date'], train['Transaction'], label='Train')
plt.plot(test['Date'], test['Transaction'], label='Test')
plt.plot(y_hat_avg['Date'],y_hat_avg['moving_avg_forecast'], label='Moving Average Forecast')
plt.legend(loc='best')
plt.show()

In [ ]:
from math import sqrt
from sklearn.metrics import mean_squared_error
rms = sqrt(mean_squared_error(test.Transaction, y_hat_avg.moving_avg_forecast))
print(rms)

### The moving average methods gives us 44.6 RMSE. Lets check the other methods

#### Exponential Smoothing (also called exponentially weighted moving average or EWMA for short)

Unlike Moving average, exponential smoothening techniques apply exponentially decreasing weights to older observations. In simple words, exponential smoothening methods give more weight to recent past observations as compared to older observations. Depending on the level of smoothening required, there may be one or more smoothening parameters to set in case of exponential smoothening.

In [ ]:
tot_tr3['ewma'] = tot_tr3['Transaction'].ewm(halflife=3, ignore_na=False,min_periods=0, adjust=True).mean()
plt.figure(figsize=(20,10))
plt.plot(tot_tr3.Transaction,'-',color='black',alpha=0.3)
plt.plot(tot_tr3.ewma,color='g')
plt.title('Transaction and Exponential Smoothening')
plt.legend()
plt.show()

In [ ]:
#from statsmodels.tsa.api import SimpleExpSmoothing, Holt
#from statsmodels.tsa.api import ExponentialSmoothing
#import statsmodels.tsa.holtwinters.ExponentialSmoothing
#y_hat_avg = test.copy()
#fit2 = SimpleExpSmoothing(np.asarray(train['Transaction'])).fit(smoothing_level=0.6,optimized=False)
#y_hat_avg['SES'] = fit2.forecast(len(test))
#plt.figure(figsize=(16,8))
#plt.plot(train['Transaction'], label='Train')
#plt.plot(test['Transaction'], label='Test')
#plt.plot(y_hat_avg['SES'], label='SES')
#plt.legend(loc='best')
#plt.show()

#### Traditional Approaches

There are matured and extensive set of modeling techniques available for time series . Out of the many, the following are a few most commonly used
and explored techniques:
* Simple moving average and exponential smoothing based forecasting
* Holt’s, Holt-Winter’s Exponential Smoothing based forecasting
* Box-Jenkins methodology (AR, MA, ARIMA, S-ARIMA, etc.)

### ARIMA 

Key Concepts

* Stationarity: One the key assumptions behind the ARIMA models. Stationarity refers to the property where for a time series its mean,
variance, and autocorrelation are time invariant. In other words, mean, variance,
and autocorrelation do not change with time. For instance, a time series having
an upward (or downward) trend is a clear indicator of a non-stationarity because
its mean would change with time. 

* Differencing: One of the methods of stationarizing series. Though there can be other
transformations, differencing is widely used to stabilize the mean of a time series. We
compute difference between consecutive observations to obtain a differenced
series. We can then apply different tests to confirm if the resulting series is stationary
or not. We can also perform second order differencing, seasonal differencing, and so
on, depending on the time series at hand.

* Unit Root Tests: Statistical tests that help us understand if a given series is stationary
or not. The Augmented Dickey Fuller test begins with a null hypothesis of series being
non-stationary, while Kwiatkowski-Phillips-Schmidt-Shin test or KPSS has a null
hypothesis that the series is stationary. We then perform a regression fit to reject or
fail to reject the null hypothesis

ARIMA stands for Auto Regressive Integrated Moving Average model.  Let’s look at the basics and constituents of this model.

* Auto Regressive or AR Modeling: A simple linear regression model where current
observation is regressed upon one or more prior observations.

* Moving Average or MA Modeling: Is again essentially a linear regression model that
models the impact of noise/error from prior observations to current one.

The ARIMA model is a logical progression and combination of the two models. Yet if we combine AR and MA with a differenced series, what we get is called as ARIMA(p,d,q) model.
where,
* p is the order of Autoregression
* q is the order of Moving average
* d is the order of differencing

Thus, for a stationary time series ARIMA models combine autoregressive and moving average concepts to model the behavior of a long running time series and helps in forecasting. Let’s now apply these concepts to transactions forecasting.

#### Since stationarity is one of the primary assumptions of ARIMA models, we will utilize Augmented Dickey Fuller test to check our series for stationarity. If the test statistic of AD Fuller test is less than the critical value(s), we reject the null hypothesis of nonstationarity.

### Lets write a custom function to compute the DF test

In [ ]:
# Dickey Fuller test for Stationarity
    
from statsmodels.tsa.stattools import adfuller
def ad_fuller_test(ts):
    dftest = adfuller(ts, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic', 'p-value','#Lags Used', 'Number of Observations Used'])
    for key,value in dftest[4].items():
        dfoutput['Critical Value (%s)'%key] = value
        print(dfoutput)

In [ ]:
# Plot rolling stats
    
def plot_rolling_stats(ts):
    a = tot_tr3['Transaction']
    ts_log = (a) 
    rolling_mean = ts_log.rolling(12).mean()
    rolling_std = ts_log.rolling(12).std()
    orig = plt.plot(ts, color='blue',label='Original')
    mean = plt.plot(rolling_mean, color='red', label='Rolling Mean')
    std = plt.plot(rolling_std, color='black', label = 'Rolling Std')
    plt.legend(loc='best')
    plt.title('Rolling Mean & Standard Deviation')
    plt.show(block=False)

In [ ]:
log_series =  (tot_tr3.Transaction.values)
log_series1 = np.log(tot_tr3.Transaction.values)
tot_tr3['log_series1'] = np.log(tot_tr3.Transaction.values)
ad_fuller_test(log_series)

In [ ]:
ad_fuller_test(log_series1)

In [ ]:
plt.figure(figsize=(18,12))
plt.plot(log_series1, 'blue', label='normal')
plt.plot(log_series, 'red', label='log')

plt.legend(loc='best')
plt.show()

#### The Test Statistic is less than the 5% critical value and therefore can be considered a stationary series. While for the log transformed data series, that isn't the case and therefore it is non-stationary.

### Plotting rolling stats

In [ ]:
from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 20, 12
a = tot_tr3['Transaction']
plot_rolling_stats(a)

The rolling mean and std seems to be stationary and we are good to go. But, 

Lets also look at a first order differenced of the log series and perform the same test of stationary

In [ ]:
log_series_shift = log_series1[1:] - log_series1[:-1]
log_series_shift = log_series_shift[~np.isnan(log_series_shift)]

In [ ]:
ad_fuller_test(log_series_shift)

In [ ]:
plt.plot(log_series_shift)

In [ ]:
plot_rolling_stats(log_series_shift)

### Again, same result as above. the differencing didn't result in a stationary series either

We still need to figure out the order of autoregression and moving average components, i.e., p and q.

One of the commonly used methods is the plotting of **ACF and PACF plots** to determine p and q values. **ACF or Auto Correlation Function plot and PACF or the Partial Auto Correlation Function** plot helps us
narrow down the search space of determining the p and q values with a few caveats. 

The ACF plot helps us understand the correlation of an observation with its lag (or previous value. The ACF plot is used to determine the MA order, i.e. q. The value at which ACF drops is the order of the MA model.
On the same lines, PACF points toward correlation between an observation and a specific lagged value, excluding effect of other lags. The value at which PACF drops points toward the order of AR model or the p in ARIMA(p,d,q)

In [ ]:
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf,plot_pacf
from statsmodels.tsa.arima_model import ARIMA, ARMA

from pandas.plotting import autocorrelation_plot
fig = plt.figure(figsize=(12,12))
ax1 = fig.add_subplot(211)
fig = sm.graphics.tsa.plot_acf(log_series_shift , lags=20, ax=ax1)
ax2 = fig.add_subplot(212)
fig = sm.graphics.tsa.plot_pacf(log_series_shift, lags=20, ax=ax2)

The ACF and PACF plot also help us understand if a series is stationary or not. If a series has gradually decreasing values for ACF and PACF, it points toward non-stationarity property in the series & here by looking at the plot above we can say p & q to be 1 & 0, & d to be 0.

Another method to derive the p, d, q parameters is to perform a grid search of the parameter space. This is more in tune with the Machine Learning way of hyperparameter tuning. 

 We need to split our dataset into train and test sets. We utilize scikit-learn’s TimeSeriesSplit utility to help us get proper training and testing sets.

In [ ]:
def auto_arima(param_max=1,series=pd.Series(),verbose=True):
    # Define the p, d and q parameters to take any value 
    # between 0 and param_max
    p = d = q = range(0, param_max+1)

    # Generate all different combinations of seasonal p, d and q triplets
    pdq = [(x[0], x[1], x[2]) for x in list(itertools.product(p, d, q))]
    
    model_resuls = []
    best_model = {}
    min_aic = 10000000
    for param in pdq:
        try:
            mod = sm.tsa.ARIMA(series, order=param)

            results = mod.fit()
            
            if verbose:
                print('ARIMA{}- AIC:{}'.format(param, results.aic))
            model_resuls.append({'aic':results.aic,
                                 'params':param,
                                 'model_obj':results})
            if min_aic>results.aic:
                best_model={'aic':results.aic,
                            'params':param,
                            'model_obj':results}
                min_aic = results.aic
        except Exception as ex:
            print(ex)
    if verbose:
        print("Best Model params:{} AIC:{}".format(best_model['params'],
              best_model['aic']))  
        
    return best_model, model_resuls

In [ ]:
import itertools
import matplotlib.dates as mdates
from sklearn.model_selection import TimeSeriesSplit

def arima_gridsearch_cv(series, cv_splits=2,verbose=True,show_plots=True):
    # prepare train-test split object
    tscv = TimeSeriesSplit(n_splits=cv_splits)
    
    # initialize variables
    splits = []
    best_models = []
    all_models = []
    i = 1
    
    # loop through each CV split
    for train_index, test_index in tscv.split(series):
        print("*"*20)
        print("Iteration {} of {}".format(i,cv_splits))
        i = i + 1
        
        # print train and test indices
        if verbose:
            print("TRAIN:", train_index, "TEST:", test_index)
        splits.append({'train':train_index,'test':test_index})
        
        # split train and test sets
        train_series = series.ix[train_index]
        test_series = series.ix[test_index]
        
        print("Train shape:{}, Test shape:{}".format(train_series.shape,
              test_series.shape))
        
        # perform auto arima
        _best_model, _all_models = auto_arima(series=train_series)
        best_models.append(_best_model)
        all_models.append(_all_models)
        
        # display summary for best fitting model
        if verbose:
            print(_best_model['model_obj'].summary())
        results = _best_model['model_obj']
        
        if show_plots:
            # show residual plots
            residuals = pd.DataFrame(results.resid)
            residuals.plot()
            plt.title('Residual Plot')
            plt.show()
            residuals.plot(kind='kde')
            plt.title('KDE Plot')
            plt.show()
            print(residuals.describe())
        
            # show forecast plot
            fig, ax = plt.subplots(figsize=(18, 4))
            fig.autofmt_xdate()
            ax = train_series.plot(ax=ax)
            test_series.plot(ax=ax)
            fig = results.plot_predict(test_series.index.min(), 
                                       test_series.index.max(), 
                                       dynamic=True,ax=ax,
                                       plot_insample=False)
            plt.title('Forecast Plot ')
            plt.legend()
            plt.show()

            # show error plot
            insample_fit = list(results.predict(train_series.index.min()+1, 
                                                train_series.index.max(),
                                                typ='levels')) 
            plt.plot((np.exp(train_series.ix[1:].tolist())-\
                             np.exp(insample_fit)))
            plt.title('Error Plot')
            plt.show()
    return {'cv_split_index':splits,
            'all_models':all_models,
            'best_models':best_models}

In [ ]:
tot_tr3c = tot_tr3.copy()
tot_tr3c = tot_tr3c.set_index('Date')
pd.to_datetime(tot_tr3c.index)
results_dict = arima_gridsearch_cv(tot_tr3c.log_series1,cv_splits=5)

### The lowest AIC p, d, q value wins and in this case it is 1, 0, 0 and looking at the forecast graph,  although it captures the trend,  it does miss out at times.

In [ ]:
model = ARIMA(log_series_shift, order=(1,0,0))  
results_AR = model.fit()
plt.plot(log_series1)
plt.plot(results_AR.fittedvalues, color='red')

### Visualizing using Prophet 

Prophet is a procedure for forecasting time series data based on an additive model where non-linear trends are fit with yearly, weekly, and daily seasonality, plus holiday effects. It works best with time series that have strong seasonal effects and several seasons of historical data. Prophet is robust to missing data and shifts in the trend, and typically handles outliers well.

In [ ]:
transactions = pd.DataFrame(tot_tr)
transactions.columns = ['ds', 'y']

In [ ]:
from fbprophet import Prophet

m = Prophet()
m.fit(transactions)
future = m.make_future_dataframe(periods=365)
forecast = m.predict(future)
forecast.head(10)

### Plot of the forecast  & the trends

In [ ]:
py.iplot([
    go.Scatter(x=transactions['ds'], y=transactions['y'], name='y'),
    go.Scatter(x=forecast['ds'], y=forecast['yhat'], name='yhat'),
    go.Scatter(x=forecast['ds'], y=forecast['yhat_upper'], fill='tonexty', mode='none', name='upper'),
    go.Scatter(x=forecast['ds'], y=forecast['yhat_lower'], fill='tonexty', mode='none', name='lower'),
    go.Scatter(x=forecast['ds'], y=forecast['trend'], name='Trend')
])

In [ ]:
m = Prophet(changepoint_prior_scale=2.5)
m.fit(transactions)
future = m.make_future_dataframe(periods=365)
forecast = m.predict(future)

In [ ]:
# Calculate root mean squared error.

print('RMSE: %f' % np.sqrt(np.mean((forecast.loc[:1682, 'yhat']-transactions['y'])**2)) )
py.iplot([
    go.Scatter(x=transactions['ds'], y=transactions['y'], name='y'),
    go.Scatter(x=forecast['ds'], y=forecast['yhat'], name='yhat'),
    go.Scatter(x=forecast['ds'], y=forecast['yhat_upper'], fill='tonexty', mode='none', name='upper'),
    go.Scatter(x=forecast['ds'], y=forecast['yhat_lower'], fill='tonexty', mode='none', name='lower'),
    go.Scatter(x=forecast['ds'], y=forecast['trend'], name='Trend')
])

### RMSE after adding seasonality

In [ ]:
m = Prophet(changepoint_prior_scale=2.5)
m.add_seasonality(name='monthly', period=30.5, fourier_order=5)
m.fit(transactions)
future = m.make_future_dataframe(periods=365)
forecast = m.predict(future)

In [ ]:
# Calculate root mean squared error.

print('RMSE: %f' % np.sqrt(np.mean((forecast.loc[:1682, 'yhat']-transactions['y'])**2)) )
py.iplot([
    go.Scatter(x=transactions['ds'], y=transactions['y'], name='y'),
    go.Scatter(x=forecast['ds'], y=forecast['yhat'], name='yhat'),
    go.Scatter(x=forecast['ds'], y=forecast['yhat_upper'], fill='tonexty', mode='none', name='upper'),
    go.Scatter(x=forecast['ds'], y=forecast['yhat_lower'], fill='tonexty', mode='none', name='lower'),
    go.Scatter(x=forecast['ds'], y=forecast['trend'], name='Trend')
])

# Association Rule-Mining using the  Apriori algorithm

**Association Analysis 101**

Although, there are many complex ways to analyze data (clustering, regression, Neural Networks, Random Forests, SVM, etc.) the challenge with many of these approaches is that they can be difficult to tune, challenging to interpret and require quite a bit of data prep and feature engineering to get good results. In other words, they can be very powerful but require a lot of knowledge to implement properly.

Association analysis is relatively light on the math concepts and easy to explain to non-technical people. In addition, it is an unsupervised learning tool that looks for hidden patterns so there is limited need for data prep and feature engineering. It is a good start for certain cases of data exploration and can point the way for a deeper dive into the data using other approaches.

Association rules are normally written like this: {Diapers} -> {Beer} which means that there is a strong relationship between customers that purchased diapers and also purchased beer in the same transaction.

In the above example, the {Diaper} is the antecedent and the {Beer} is the consequent. Both antecedents and consequents can have multiple items. In other words, {Diaper, Gum} -> {Beer, Chips} is a valid rule.

Here are the  key metrics to consider when evaluating association rules:
***
**Support** is the relative frequency that the rules show up. In many instances, you may want to look for high support in order to make sure it is a useful relationship. However, there may be instances where a low support is useful if you are trying to find “hidden” relationships.

support(A→C)=support(A∪C),range: [0,1]

**Confidence** is a measure of the reliability of the rule. A confidence of .5 in the above example would mean that in 50% of the cases where Diaper and Gum were purchased, the purchase also included Beer and Chips. For product recommendation, a 50% confidence may be perfectly acceptable but in a medical situation, this level may not be high enough.

Confidence(A→C)=support(A→C)support(A),range: [0,1]

**Lift** is the ratio of the observed support to that expected if the two rules were independent (see wikipedia). The basic rule of thumb is that a lift value close to 1 means the rules were completely independent. Lift values > 1 are generally more “interesting” and could be indicative of a useful rule pattern.

lift(A→C)=confidence(A→C)support(C),range: [0,∞]

**leverage:**
levarage(A→C)=support(A→C)−support(A)×support(C),range: [−1,1]

Leverage computes the difference between the observed frequency of A and C appearing together and the frequency that would be expected if A and C were independent. An leverage value of 0 indicates independence.

**conviction':**
conviction(A→C)=1−support(C)1−confidence(A→C),range: [0,∞]

A high conviction value means that the consequent is highly depending on the antecedent. For instance, in the case of a perfect confidence score, the denominator becomes 0 (due to 1 - 1) for which the conviction score is defined as 'inf'. Similar to lift, if items are independent, the conviction is 1.
***

In [ ]:
item_cnt = data['Item'].value_counts().reset_index()
item_cnt.columns = ['Item', 'Count']
item_cnt = item_cnt[item_cnt.Item != 'NONE']

In [ ]:
objects = (list(item_cnt['Item'].head(n=20)))
y_pos = np.arange(len(objects))
performance = list(item_cnt['Count'].head(n=20))
plt.bar(y_pos, performance, align='center', alpha=0.5)
plt.xticks(y_pos, objects, rotation='vertical')
plt.ylabel('Item count')
plt.title('Item Sales distribution')

In [ ]:
total_item_count = item_cnt['Item'].count()
item_cnt['item_perc'] = item_cnt['Count']/total_item_count
item_cnt['total_perc'] = item_cnt.item_perc.cumsum()
ict = item_cnt.head(10)

In [ ]:
import plotly.graph_objs as go

trace = go.Table(
    header=dict(values=['Item', 'Count', 'Item Perc', 'Total Perc'],
                line = dict(color='#7D7F80'),
                fill = dict(color='#a1c3d1'),
                align = ['left'] * 5),
    cells=dict(values=[ict['Item'].values.tolist(),
                       ict['Count'].values.tolist(),
                       ict['item_perc'].values.tolist(),
                       ict['total_perc'].values.tolist()],
               line = dict(color='#7D7F80'),
               fill = dict(color='#EDFAFF'),
               align = ['left'] * 5))

layout = dict(width=600, height=600)
data1 = [trace]
fig = dict(data=data1, layout=layout)
py.iplot(fig, filename = 'styled_table')

In [ ]:
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

### We need the list of items from the same transaction as a list so we can form associations

In [ ]:
items = []
for i in data['Transaction'].unique():
    itemlist = list(set(data[data["Transaction"]==i]['Item']))
    if len(itemlist) > 0:
        items.append(itemlist)

In [ ]:
### We need to one-hot encode the items as the library we are working with accepts only 1,0, True or False

from mlxtend.preprocessing import TransactionEncoder

oht = TransactionEncoder()
oht_item = oht.fit(items).transform(items)
df1 = pd.DataFrame(oht_item, columns=oht.columns_)
frequent_itemset = apriori(df1, use_colnames=True, min_support=0.02)
rules = association_rules(frequent_itemset, metric="lift", min_threshold=0.5)

In [ ]:
rules.head(10)

### Looking at the above table, we can say that cake, bread & coffee are bought together often

### Lets visualize support and confidence

In [ ]:
plt.figure(figsize=(12,12))
plt.scatter(rules['support'],rules['confidence'],marker='*',edgecolors='grey',s=100,c=rules['lift'])
plt.colorbar(label='Lift')
plt.xlabel('support')
plt.ylabel('confidence')

### Lets visualize the network graph of the associated items

In [ ]:
def draw_graph(rules, rules_to_show):
    import networkx as nx  
    plt.figure(figsize=(10,8))
    G1 = nx.DiGraph()
    
    color_map=[]
    N = 400
    colors = np.random.rand(N)    
    strs=[]
    for i in range(rules_to_show):
        strs.append('R'+str(i))
    
    for i in range (rules_to_show):      
        G1.add_nodes_from(["R"+str(i)])
         
        for a in rules.iloc[i]['antecedents']:
                
            G1.add_nodes_from([a])        
            G1.add_edge(a, "R"+str(i), color=colors[i] , weight = 1.5)
        
        for c in rules.iloc[i]['consequents']:         
            G1.add_nodes_from([c])
            G1.add_edge("R"+str(i), c, color=colors[i],  weight=1.5)
    
    for node in G1:
        if node in strs:
            color_map.append('black')
        else:
            color_map.append('red')
            
    edges = G1.edges()
    colors = [G1[u][v]['color'] for u,v in edges]
    weights = [G1[u][v]['weight'] for u,v in edges]

    pos = nx.spring_layout(G1, k=16, scale=1)
    nx.draw(G1, pos, edges=edges, node_color = color_map, edge_color=colors, width=weights, font_size=14, with_labels=False)            
    for p in pos:  # raise text positions
        pos[p][1] += 0.08
    nx.draw_networkx_labels(G1, pos)

In [ ]:
draw_graph(rules,len(rules))

### How do you interpret the above network graph?

The R1, R2 etc..  are the row numbers in the rules dataframe and the arrows pointing to and from the corresponding nodes are the antecedents
and consequents of that row. 